# 05. 듀얼 커브: KOFR 할인 + CD91 예측, 지수, IRS

같은 '금리 커브'라도 시세 종류와 관례에 따라 조립이 다르다. 그 차이는 **구간 종류별 헬퍼 함수**(`HELPERS`)와 **커브 사이 의존**(할인 커브를 먼저)으로 담는다.

| 단계 | 코드 |
|---|---|
| 커브 레시피 | `configs/market.yaml` (curves.KOFR, curves.CD91) |
| 지수 (파생 객체) | `configs/market.yaml` (indices), `config.IndexConfig.make`, `Market.index` |
| 헬퍼 사전·의존 순서 | `curves.HELPERS`, `Config.build_order` |
| 스왑·엔진 | `portfolio.IRSwap`, `portfolio.ENGINES`, `configs/products.yaml` (engines) |

> 데이터: `tests/fixtures/market_sample.txt`는 **합성 샘플**이다. 흐름을 익히는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

from quanttool.config import load_config
from quanttool.market import build_market
from quanttool.market.quotes import read_quotes
from quanttool.portfolio import Portfolio

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
cfg = load_config(ROOT / "configs")
quotes = read_quotes(ROOT / "tests" / "fixtures" / "market_sample.txt")  # 합성 샘플
ASOF = date(2026, 10, 23)
market = build_market(ASOF, quotes[ASOF], cfg)

cfg.build_order()

## 1. 조립 순서
CD91 커브는 스왑 헬퍼의 할인에 KOFR를 쓰고(`discount_curve: KOFR`), 지수 CD91의 예측 커브는 자기 자신이다. 그래서 KOFR가 먼저 만들어진다. 순환이 있으면 `build_order`가 오류를 낸다.

In [ ]:
(
    {n: sorted(c.dependencies(cfg.indices)) for n, c in cfg.curves.items()},
    [(s.type, s.index, s.tenors) for s in cfg.curves["CD91"].segments],
)

## 2. 진단: 헬퍼 재가격 (bp)
듀얼 커브여도 각 헬퍼가 자기 시세를 다시 낸다.

In [ ]:
from quanttool.market.curves import helper_repricing

pd.concat({n: helper_repricing(market.built[n], ASOF) for n in ["KOFR", "CD91"]})

## 3. 커브 비교: CD91 − KOFR 베이시스

In [ ]:
from quanttool.market.curves import conventions_for, curve_table

tenors = ["3M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]
with market.active():
    t = {n: curve_table(market[n], conventions_for(cfg, n), tenors) for n in ["KOFR", "CD91"]}
basis = (t["CD91"]["zero_cc"] - t["KOFR"]["zero_cc"]) * 1e4
pd.DataFrame({"KOFR": t["KOFR"]["zero_cc"], "CD91": t["CD91"]["zero_cc"], "basis_bp": basis})

## 4. 지수는 예측 커브에 연결된 파생 객체
`market.index('CD91')`의 선도금리는 CD91 커브(요인 포함)에서 나온다. CD91 요인을 흔들면 지수 선도가 따라오고, KOFR만 흔들면 그대로다.

주의: 커브는 오늘 시세로 **얼려** 있다. KOFR를 흔들어도 CD91 커브를 다시 부트스트랩하지 않는다 (시세에 연결해 다시 풀려면 `build_market(..., linked=True)`로 만들고 시세 Quote를 바꾼다).

In [ ]:
from quanttool.analysis.scenarios import parallel

cal = ql.SouthKorea(ql.SouthKorea.Settlement)
fix = cal.adjust(ql.Date(23, 10, 2028))
idx = market.index("CD91")
rows = {}
for name, s in {
    "base": {},
    "CD91 +10bp": parallel(market, "CD91", 1e-3),
    "KOFR +10bp": parallel(market, "KOFR", 1e-3),
}.items():
    with market.active(), market.scenario(s):
        rows[name] = idx.fixing(fix)
pd.Series(rows, name="CD91 선도 (2028-10)")

## 5. IRS: 상품은 조립 함수, 엔진은 설정
`tests/fixtures/krw_swaps.yaml`의 거래는 `index: CD91`, `discount: KOFR`로 시장 객체를 이름으로 가리킨다. 엔진은 `configs/products.yaml` engines.defaults의 `IRSwap: DiscountingSwap`.

In [ ]:
book = Portfolio.from_file(
    ROOT / "tests" / "fixtures" / "krw_swaps.yaml", products=cfg.products
).build(market, cfg)
with market.active():
    fair = {i: s.fairRate() for i, s in book.instruments.items()}
print("엔진:", list(book.engines))
pd.DataFrame(
    {
        "npv": book.npvs(),
        "fair_rate": fair,
        "fixed_rate": {i: t.fixed_rate for i, t in book.trades.items()},
    }
)

par 스왑(고정금리 = 시세)은 NPV 0이다 — 커브가 시세를 재현한다는 또 다른 확인.

In [ ]:
s = cal.advance(ql.Date(23, 10, 2026), 1, ql.Days)
start = date(s.year(), s.month(), s.dayOfMonth())
par = Portfolio.from_records(
    [
        dict(
            id="par5Y",
            type="IRSwap",
            notional=1e10,
            fixed_rate=quotes[ASOF]["IRS/RATE/KRW/CD91/5Y"],
            start_date=start,
            maturity_date=start.replace(year=start.year + 5),
            conventions="KRW_IRS",
            index="CD91",
            discount="KOFR",
        )
    ]
)
par.build(market, cfg).npv()

## 6. key rate: 할인과 예측이 따로
같은 스왑이라도 KOFR 요인(할인)과 CD91 요인(예측)의 민감도가 따로 나온다. 고정 지급은 CD91 상승에 이익, 할인 민감도는 순 현금흐름 크기만큼 작다.

In [ ]:
from quanttool.analysis.risk import key_rate

kr = pd.concat({c: key_rate(book, market, cfg.risk, c)["delta"] for c in ["CD91", "KOFR"]}, axis=1)
kr.reindex(market.objects["KOFR"].tenors).fillna(0).round(0)